# **SovRoute × CrewAI: Pre-Flight Bitcoin & EVM Security Guardrail (Remote MCP)**

This quickstart demonstrates connecting CrewAI agents to **SovRoute**'s Remote Streamable HTTP MCP server (`https://node.sovroute.com/mcp`) as a **deterministic pre-flight security guardrail** before accepting 0-conf payments, broadcasting transactions, or signing EVM delegations.

### 7 Deterministic Read-Only & Zero-Broadcast Safety Tools
No wallet or API key setup is required for demo evaluation (**15 free evaluation calls/24h per IP**, then Pay-Per-Call via **L402 Lightning (1–5 sats)** or **x402 Base USDC ($0.001–$0.005)** — no accounts or API keys):
1. `get_bitcoin_smart_fees` — Real-time Bitcoin Core v31.1 fee projections (`sat/vB`)
2. `get_mempool_entropy` — Live mempool congestion, tx count, vsize, and pending fee pool
3. `get_next_block_xray` — Live miner candidate block template (`getblocktemplate`) cutoff fee & tx queue rank
4. `check_rbf_fraud_shield` — 0-conf BIP-125 Replace-By-Fee double-spend risk detection
5. `simulate_raw_transaction` — Zero-broadcast `testmempoolaccept` consensus dry-run (`broadcastedToNetwork: false`)
6. `verify_block_header` — Latest mined Bitcoin block header truth
7. `inspect_eip7702_authorization` — Deterministic EIP-7702 opcode/selector & cross-chain replay scanner

In [ ]:
%pip install -q "crewai>=1.0.0" "crewai-tools[mcp]" httpx python-dotenv

## 1. Key-Free Smoke Test (Live MCP Handshake, Tool Discovery & Smart Fees)
Verify connectivity to `https://node.sovroute.com/mcp` over Streamable HTTP without any LLM or wallet key.

In [1]:
import json
import httpx

MCP_URL = "https://node.sovroute.com/mcp"
HEADERS = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}

def parse_mcp(text: str) -> dict:
    for line in text.splitlines():
        if line.startswith("data:"):
            return json.loads(line[5:].strip())
    return json.loads(text)

with httpx.Client(timeout=15.0) as client:
    client.post(MCP_URL, headers=HEADERS, json={
        "jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {"protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "crewai-quickstart", "version": "1.0.0"}}
    }).raise_for_status()

    tools_res = client.post(MCP_URL, headers=HEADERS, json={"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}})
    tools_res.raise_for_status()
    tools = parse_mcp(tools_res.text)["result"]["tools"]
    print(f"Discovered {len(tools)} tools:", [t["name"] for t in tools])

    fee_res = client.post(MCP_URL, headers=HEADERS, json={
        "jsonrpc": "2.0", "id": 3, "method": "tools/call",
        "params": {"name": "get_bitcoin_smart_fees", "arguments": {}}
    })
    fee_res.raise_for_status()
    print("Live fees:", parse_mcp(fee_res.text)["result"]["content"][0]["text"])

Discovered 7 tools: ['get_bitcoin_smart_fees', 'get_mempool_entropy', 'check_rbf_fraud_shield', 'get_next_block_xray', 'simulate_raw_transaction', 'verify_block_header', 'inspect_eip7702_authorization']
Live fees: {
  "fastestSatPerVb": 1.3,
  "halfHourSatPerVb": 1,
  "hourSatPerVb": 1,
  "economicSatPerVb": 1,
  "targetBlocksFastest": 2,
  "targetBlocksEconomic": 6
}


## 2. CrewAI 1.x Native 1-Line DSL (`mcps=[...]`) Pre-Flight Security Guardrail
Pass `https://node.sovroute.com/mcp` directly into `Agent(..., mcps=[...])` to audit a live transaction and produce an `APPROVE`, `HOLD`, or `REJECT` verdict.

In [ ]:
import os
from crewai import Agent, Crew, Task

if not os.getenv("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY in your environment to run the full CrewAI Security Analyst agent.")
else:
    security_analyst = Agent(
        role="Bitcoin & EVM Pre-Flight Security Analyst",
        goal="Verify transaction safety using bare-metal Bitcoin Core v31.1 and opcode forensics before settlement.",
        backstory="Deterministic blockchain risk officer that never guesses fees, RBF signals, or consensus validity.",
        mcps=["https://node.sovroute.com/mcp"],
        verbose=True,
    )

    audit_task = Task(
        description=(
            "1. Call get_bitcoin_smart_fees and get_mempool_entropy to inspect live congestion.\n"
            "2. Call get_next_block_xray to check candidate block fill and cutoff fee rate.\n"
            "3. Produce a concise Pre-Flight Security Report ending with APPROVE, HOLD, or REJECT."
        ),
        expected_output="Structured Markdown Pre-Flight Security Report with a clear final verdict.",
        agent=security_analyst,
    )

    crew = Crew(agents=[security_analyst], tasks=[audit_task], verbose=True)
    result = crew.kickoff()
    print(result)